# 📊 Pocket OTC AI Image Analyzer — Google Colab

يشغّل واجهة الويب العامة وTelegram Bot معًا.

**READ-ONLY:** لا يسجل الدخول إلى Pocket Option ولا ينفذ أوامر تداول.

In [ ]:
!rm -rf /content/Jjjjjjj
!git clone -q https://github.com/mohmb142/Jjjjjjj.git /content/Jjjjjjj
%cd /content/Jjjjjjj
!python -m pip install -q -r colab_requirements.txt
print('✅ تم تنزيل المشروع وتثبيت المتطلبات')

In [ ]:
import os
from getpass import getpass

telegram_token = getpass('🔐 Telegram Bot Token: ').strip()
openrouter_key = getpass('🔐 OpenRouter API Key: ').strip()

if not telegram_token:
    raise ValueError('Telegram Bot Token مطلوب')
if not openrouter_key:
    raise ValueError('OpenRouter API Key مطلوب')

os.environ['TELEGRAM_BOT_TOKEN'] = telegram_token
os.environ['OPENROUTER_API_KEY'] = openrouter_key
os.environ['OPENROUTER_MODEL'] = 'google/gemini-2.5-flash'

print('✅ تم إعداد المفاتيح داخل جلسة Colab فقط')

In [ ]:
# تشغيل Web UI + Telegram Bot معًا
import threading
import time
import socket
import uvicorn
from google.colab.output import eval_js
from IPython.display import display, HTML

PORT = 8000

def port_ready(host='127.0.0.1', port=PORT, timeout=15):
    end = time.time() + timeout
    while time.time() < end:
        try:
            with socket.create_connection((host, port), timeout=0.5):
                return True
        except OSError:
            time.sleep(0.25)
    return False

def web_worker():
    try:
        uvicorn.run('main:app', host='0.0.0.0', port=PORT, log_level='warning')
    except Exception as exc:
        print('❌ Web UI error:', exc)

web_thread = threading.Thread(target=web_worker, daemon=True, name='fastapi-web')
web_thread.start()

if not port_ready():
    raise RuntimeError('لم تبدأ FastAPI على المنفذ 8000')

public_url = eval_js(f'google.colab.kernel.proxyPort({PORT})')

print('🌐 Web UI: تعمل')
print('🔗 الرابط العام:', public_url)

display(HTML(
    f'<div style="padding:20px;font-family:Arial">'
    f'<h2>📊 Pocket OTC AI Analyzer</h2>'
    f'<p>واجهة الويب تعمل الآن.</p>'
    f'<a href="{public_url}" target="_blank" '
    f'style="display:inline-block;padding:14px 22px;background:#1976d2;color:#fff;text-decoration:none;border-radius:8px">'
    f'🚀 فتح الواجهة</a></div>'
))

# تشغيل Telegram في Thread مستقل
from telegram_bot import run

def telegram_worker():
    try:
        print('🤖 بدء Telegram Bot...')
        run()
    except Exception as exc:
        print('❌ Telegram Bot error:', exc)

telegram_thread = threading.Thread(target=telegram_worker, daemon=True, name='telegram-bot')
telegram_thread.start()

print('🤖 Telegram Bot: يعمل في الخلفية')
print('✅ Web UI + Telegram يعملان معًا')

In [ ]:
# اختبار الواجهة
import requests

try:
    response = requests.get('http://127.0.0.1:8000/', timeout=10)
    print('HTTP status:', response.status_code)
    if response.status_code == 200:
        print('✅ اختبار Web UI نجح')
    else:
        print('⚠️ Web UI تعمل ولكن الحالة:', response.status_code)
except Exception as exc:
    print('❌ فشل اختبار Web UI:', exc)

print('')
print('🌐 الرابط العام:', public_url)
print('⚠️ لا تغلق جلسة Colab حتى يبقى الرابط والبوت يعملان.')